# Entity Identification Pipeline for Co-Pilot Agent

## Overview
This notebook compares two approaches for predicting relevant entities from user queries:

1. **Pre-trained Classifier (Zero-shot)**: Use a pre-trained model without any fine-tuning on our data
2. **Fine-tuned Classifier**: Train a model on our training data

**Important**: No data leakage - test set is never seen during training.

## Entity Types
- CDR (Call Detail Records)
- Phone
- Web Activity
- Web Actor
- Person
- Investigation
- Insight
- Report
- EVisa Request

## 1. Setup and Dependencies

In [ ]:
# Install required packages
# !pip install pandas numpy scikit-learn torch transformers accelerate

import pandas as pd
import numpy as np
import json
import ast
from typing import List, Dict, Tuple
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

# Deep learning imports
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Load and Preprocess Data

In [ ]:
# Load datasets
user_queries_df = pd.read_csv('user_queries.csv')
fields_description_df = pd.read_csv('fields_description.csv')

print(f"User Queries: {len(user_queries_df)} rows")
print(f"Fields Description: {len(fields_description_df)} rows")

In [ ]:
def safe_parse_json(json_str: str) -> dict:
    """Parse JSON string, handling Python dict format."""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except (ValueError, SyntaxError):
            return {}

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType keys."""
    entities = set()
    
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search_statements(statements):
        if not statements:
            return
        for stmt in statements:
            if isinstance(stmt, dict):
                params = stmt.get('parameters', {})
                if 'relationTargetType' in params:
                    targets = params['relationTargetType']
                    if isinstance(targets, list):
                        entities.update(targets)
                    else:
                        entities.add(targets)
                if 'statements' in stmt:
                    search_statements(stmt['statements'])
    
    if 'statements' in json_obj:
        search_statements(json_obj['statements'])
    
    return sorted(list(entities))

# Parse and extract entities
user_queries_df['parsed_json'] = user_queries_df['json'].apply(safe_parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities)

# Show distribution
all_entities_flat = [e for ents in user_queries_df['entities'] for e in ents]
entity_counts = Counter(all_entities_flat)
print("Entity Distribution:")
for entity, count in entity_counts.most_common():
    print(f"  {entity}: {count} ({100*count/len(user_queries_df):.1f}%)")

multi_entity = sum(1 for e in user_queries_df['entities'] if len(e) > 1)
print(f"\nQueries with multiple entities: {multi_entity} ({100*multi_entity/len(user_queries_df):.1f}%)")

In [ ]:
# Define entity labels with descriptions for zero-shot classification
ALL_ENTITIES = sorted(list(set(all_entities_flat)))
print(f"All entity types ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")

# Entity descriptions for zero-shot (helps the model understand what each entity means)
ENTITY_DESCRIPTIONS = {
    'CDR': 'Call Detail Records including phone calls, SMS messages, emails, and communications',
    'EVisa Request': 'Electronic visa applications and travel document requests',
    'Insight': 'Intelligence insights and analysis notes',
    'Investigation': 'Investigation cases and inquiries',
    'Person': 'Individual people with personal information like name, birth date, occupation',
    'Phone': 'Phone devices and identifiers like IMEI, IMSI, MSISDN, phone numbers',
    'Report': 'Reports and documentation',
    'Web Activity': 'Social media posts, comments, tweets, and online content',
    'Web Actor': 'Social media profiles and accounts on platforms like Facebook, Twitter, Instagram'
}

# Labels for zero-shot classification (can use descriptions or simple names)
ZERO_SHOT_LABELS = list(ENTITY_DESCRIPTIONS.values())
LABEL_TO_ENTITY = {desc: entity for entity, desc in ENTITY_DESCRIPTIONS.items()}

## 3. Train/Test Split (No Data Leakage)

In [ ]:
# Create label encoder
mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

# Split: 80% train, 20% test - STRICT SEPARATION
X = user_queries_df['question'].tolist()
y = y_encoded
entities_list = user_queries_df['entities'].tolist()

indices = list(range(len(X)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# Training data - ONLY used for fine-tuning (Approach 2)
X_train = [X[i] for i in train_idx]
y_train = y[train_idx]
train_entities = [entities_list[i] for i in train_idx]

# Test data - used for evaluation of BOTH approaches
X_test = [X[i] for i in test_idx]
y_test = y[test_idx]
test_entities = [entities_list[i] for i in test_idx]

print(f"Train size: {len(X_train)} (used ONLY for fine-tuning)")
print(f"Test size: {len(X_test)} (used for evaluation of both approaches)")
print(f"\n*** No data leakage: Test data is never seen during training ***")

## 4. Evaluation Framework

In [ ]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, 
                        label_names: List[str], method_name: str = "Method") -> Dict:
    """Compute and display evaluation metrics."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'jaccard_micro': jaccard_score(y_true, y_pred, average='micro', zero_division=0),
    }
    
    print(f"\n{'='*60}")
    print(f"RESULTS: {method_name}")
    print(f"{'='*60}")
    print(f"Exact Match Accuracy: {metrics['exact_match']:.4f}")
    print(f"F1 Micro:             {metrics['f1_micro']:.4f}")
    print(f"F1 Macro:             {metrics['f1_macro']:.4f}")
    print(f"Precision Micro:      {metrics['precision_micro']:.4f}")
    print(f"Recall Micro:         {metrics['recall_micro']:.4f}")
    print(f"Jaccard Micro:        {metrics['jaccard_micro']:.4f}")
    print(f"Hamming Loss:         {metrics['hamming_loss']:.4f}")
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    
    return metrics

---
# APPROACH 1: Pre-trained Classifier (Zero-Shot)

Use a pre-trained NLI model for zero-shot classification.
**No training on our data** - the model has never seen our queries or labels.

In [ ]:
class ZeroShotEntityClassifier:
    """
    Zero-shot entity classifier using pre-trained NLI model.
    No fine-tuning - uses the model as-is.
    """
    
    def __init__(self, model_name: str = "facebook/bart-large-mnli"):
        """
        Initialize with a pre-trained zero-shot classification model.
        
        Options:
        - facebook/bart-large-mnli (best quality, larger)
        - cross-encoder/nli-deberta-v3-small (smaller, faster)
        - typeform/distilbert-base-uncased-mnli (balanced)
        """
        self.model_name = model_name
        print(f"Loading pre-trained model: {model_name}")
        print("*** No fine-tuning - using model as-is ***")
        
        self.classifier = pipeline(
            "zero-shot-classification",
            model=model_name,
            device=0 if torch.cuda.is_available() else -1
        )
        print("Model loaded!")
        
        # Store entity mappings
        self.entity_labels = ALL_ENTITIES
        self.label_descriptions = ENTITY_DESCRIPTIONS
    
    def predict_single(self, query: str, threshold: float = 0.3, 
                       use_descriptions: bool = True, multi_label: bool = True) -> Tuple[List[str], Dict[str, float]]:
        """
        Predict entities for a single query using zero-shot classification.
        
        Args:
            query: The user query
            threshold: Minimum confidence to include an entity
            use_descriptions: Use detailed descriptions instead of entity names
            multi_label: Allow multiple entity predictions
        """
        # Use descriptions or simple entity names
        if use_descriptions:
            candidate_labels = list(self.label_descriptions.values())
        else:
            candidate_labels = self.entity_labels
        
        # Run zero-shot classification
        result = self.classifier(
            query,
            candidate_labels=candidate_labels,
            multi_label=multi_label
        )
        
        # Map back to entity names and filter by threshold
        predictions = []
        scores = {}
        
        for label, score in zip(result['labels'], result['scores']):
            # Map description back to entity name if needed
            if use_descriptions:
                entity = LABEL_TO_ENTITY.get(label, label)
            else:
                entity = label
            
            scores[entity] = score
            if score >= threshold:
                predictions.append(entity)
        
        # If no predictions above threshold, take the top one
        if not predictions:
            top_label = result['labels'][0]
            if use_descriptions:
                predictions = [LABEL_TO_ENTITY.get(top_label, top_label)]
            else:
                predictions = [top_label]
        
        return predictions, scores
    
    def predict_batch(self, queries: List[str], threshold: float = 0.3,
                      use_descriptions: bool = True, verbose: bool = True) -> List[List[str]]:
        """
        Predict entities for multiple queries.
        """
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 20 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred, _ = self.predict_single(query, threshold, use_descriptions)
            predictions.append(pred)
        return predictions

In [ ]:
# Initialize zero-shot classifier
print("="*60)
print("APPROACH 1: Pre-trained Classifier (Zero-Shot)")
print("="*60)
print("\nThis model has NEVER seen our training data.")
print("It uses general language understanding to classify.\n")

# Use a smaller model for faster inference, or bart-large-mnli for best quality
zero_shot_classifier = ZeroShotEntityClassifier(
    model_name="facebook/bart-large-mnli"  # or "typeform/distilbert-base-uncased-mnli" for faster
)

In [ ]:
# Test zero-shot on a few examples
print("\nTesting zero-shot classifier:")
test_queries_sample = [
    "What SMS messages were sent from suspicious phones?",
    "Find all individuals with occupation engineer",
    "Show me tweets from accounts mentioning Tesla"
]

for query in test_queries_sample:
    pred, scores = zero_shot_classifier.predict_single(query, threshold=0.3)
    print(f"\nQuery: {query}")
    print(f"Predicted: {pred}")
    print(f"Top scores: {dict(sorted(scores.items(), key=lambda x: -x[1])[:3])}")

In [ ]:
# Run zero-shot predictions on TEST SET
print("\nRunning zero-shot predictions on test set...")
print("(This may take a while as each query requires a forward pass)\n")

predictions_zeroshot = zero_shot_classifier.predict_batch(X_test, threshold=0.3)
y_pred_zeroshot = mlb.transform(predictions_zeroshot)

metrics_zeroshot = evaluate_predictions(
    y_test, y_pred_zeroshot, ALL_ENTITIES,
    "Approach 1: Pre-trained (Zero-Shot)"
)

---
# APPROACH 2: Fine-tuned Classifier

Train a classifier on our training data.
**Uses training data** - but test data is never seen during training.

In [ ]:
class EntityDataset(Dataset):
    """PyTorch Dataset for entity classification."""
    
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }


class FineTunedEntityClassifier:
    """
    Fine-tuned classifier for entity prediction.
    Trained on training data only - test data never seen.
    """
    
    def __init__(self, model_name: str = "distilbert-base-uncased", num_labels: int = 9):
        self.model_name = model_name
        self.num_labels = num_labels
        self.device = DEVICE
        self.model = None
        self.tokenizer = None
        self.label_names = ALL_ENTITIES
    
    def train(self, X_train: List[str], y_train: np.ndarray,
              epochs: int = 10, batch_size: int = 16, max_length: int = 128):
        """
        Train the classifier on training data ONLY.
        Test data is never used here.
        """
        print(f"\n{'='*60}")
        print("TRAINING: Fine-tuned Classifier")
        print(f"{'='*60}")
        print(f"Model: {self.model_name}")
        print(f"Training samples: {len(X_train)}")
        print(f"Epochs: {epochs}")
        print("\n*** Test data is NOT used during training ***")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification"
        )
        
        # Split TRAINING data for train/validation (NOT using test data)
        X_t, X_v, y_t, y_v = train_test_split(
            X_train, y_train, test_size=0.15, random_state=42
        )
        
        print(f"Training split: {len(X_t)} train, {len(X_v)} validation")
        
        train_dataset = EntityDataset(X_t, y_t, self.tokenizer, max_length)
        val_dataset = EntityDataset(X_v, y_v, self.tokenizer, max_length)
        
        training_args = TrainingArguments(
            output_dir='./classifier_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
        )
        
        print("\nTraining...")
        trainer.train()
        print("Training complete!")
        
        self.model.eval()
        self.model.to(self.device)
    
    def predict_single(self, query: str, threshold: float = 0.5) -> Tuple[List[str], np.ndarray]:
        """Predict entities for a single query."""
        inputs = self.tokenizer(
            query,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        
        predicted = [self.label_names[i] for i, p in enumerate(probs) if p > threshold]
        return predicted if predicted else [self.label_names[np.argmax(probs)]], probs
    
    def predict_batch(self, queries: List[str], threshold: float = 0.5,
                      verbose: bool = True) -> List[List[str]]:
        """Predict entities for multiple queries."""
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 50 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred, _ = self.predict_single(query, threshold)
            predictions.append(pred)
        return predictions
    
    def save(self, path: str = './entity_classifier_model'):
        """Save the model."""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")
    
    def load(self, path: str = './entity_classifier_model'):
        """Load a saved model."""
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path)
        self.model.eval()
        self.model.to(self.device)
        print(f"Model loaded from {path}")

In [ ]:
# Initialize and train the fine-tuned classifier
print("="*60)
print("APPROACH 2: Fine-tuned Classifier")
print("="*60)

finetuned_classifier = FineTunedEntityClassifier(num_labels=len(ALL_ENTITIES))

# Train ONLY on training data (X_train, y_train)
# Test data (X_test, y_test) is NEVER seen during training
finetuned_classifier.train(X_train, y_train, epochs=10, batch_size=16)

In [ ]:
# Save the fine-tuned model
finetuned_classifier.save('./entity_classifier_model')

In [ ]:
# Run fine-tuned predictions on TEST SET (never seen during training)
print("\nRunning fine-tuned predictions on test set...")
print("(Test data was NEVER seen during training)\n")

predictions_finetuned = finetuned_classifier.predict_batch(X_test, threshold=0.5)
y_pred_finetuned = mlb.transform(predictions_finetuned)

metrics_finetuned = evaluate_predictions(
    y_test, y_pred_finetuned, ALL_ENTITIES,
    "Approach 2: Fine-tuned Classifier"
)

---
## 5. Results Comparison

In [ ]:
print("\n" + "="*70)
print("COMPARISON: Pre-trained (Zero-Shot) vs Fine-tuned")
print("="*70)

comparison_df = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Jaccard', 'Hamming Loss'],
    'Pre-trained (Zero-Shot)': [
        metrics_zeroshot['exact_match'],
        metrics_zeroshot['f1_micro'],
        metrics_zeroshot['f1_macro'],
        metrics_zeroshot['precision_micro'],
        metrics_zeroshot['recall_micro'],
        metrics_zeroshot['jaccard_micro'],
        metrics_zeroshot['hamming_loss']
    ],
    'Fine-tuned': [
        metrics_finetuned['exact_match'],
        metrics_finetuned['f1_micro'],
        metrics_finetuned['f1_macro'],
        metrics_finetuned['precision_micro'],
        metrics_finetuned['recall_micro'],
        metrics_finetuned['jaccard_micro'],
        metrics_finetuned['hamming_loss']
    ]
})

# Calculate improvement
comparison_df['Improvement'] = comparison_df['Fine-tuned'] - comparison_df['Pre-trained (Zero-Shot)']
comparison_df['Better'] = comparison_df.apply(
    lambda row: 'Fine-tuned' if (row['Improvement'] > 0 and row['Metric'] != 'Hamming Loss') or 
                                (row['Improvement'] < 0 and row['Metric'] == 'Hamming Loss') 
                else ('Zero-Shot' if row['Improvement'] != 0 else 'Tie'), axis=1
)

print(comparison_df.to_string(index=False))

print("\n" + "-"*70)
print("SUMMARY:")
ft_wins = sum(1 for b in comparison_df['Better'] if b == 'Fine-tuned')
zs_wins = sum(1 for b in comparison_df['Better'] if b == 'Zero-Shot')
print(f"  Fine-tuned wins on {ft_wins}/{len(comparison_df)} metrics")
print(f"  Zero-Shot wins on {zs_wins}/{len(comparison_df)} metrics")

f1_improvement = metrics_finetuned['f1_micro'] - metrics_zeroshot['f1_micro']
print(f"\n  F1 Micro improvement: {f1_improvement:+.4f} ({100*f1_improvement/max(metrics_zeroshot['f1_micro'], 0.001):+.1f}%)")

---
## 6. Test Cases Comparison

In [ ]:
# Define test cases
test_cases = [
    ("What SMS messages were sent from suspicious phones to 0549876543 containing 'urgent'?", ["CDR", "Phone"]),
    ("Find all calls made using 3G technology", ["CDR"]),
    ("Show me all tweets from accounts with 500 friends mentioning Tesla", ["Web Activity", "Web Actor"]),
    ("Which phones have been marked as suspicious?", ["Phone"]),
    ("Find all individuals with occupation 'engineer' born before July 1985", ["Person"]),
    ("Show me investigations that are open or created in the last 3 months", ["Investigation"]),
    ("Find insights containing 'money laundering' from the past month", ["Insight"]),
    ("List visitors whose travel document was issued before January 2020", ["EVisa Request"]),
    ("Get reports created in the past 3 days", ["Report"]),
    ("Find Instagram profiles with 100 followers using Israel phone number", ["Web Actor"]),
    ("List emails sent to phones associated with target Sarah Johnson", ["CDR", "Phone"]),
]

print("\n" + "="*70)
print("TEST CASES COMPARISON")
print("="*70)

def check_prediction(pred, expected):
    """Check if prediction matches expected."""
    pred_set, exp_set = set(pred), set(expected)
    if pred_set == exp_set:
        return "✓ EXACT"
    elif pred_set & exp_set:
        return "~ PARTIAL"
    else:
        return "✗ WRONG"

zs_correct = 0
ft_correct = 0

for query, expected in test_cases:
    # Zero-shot prediction
    zs_pred, _ = zero_shot_classifier.predict_single(query, threshold=0.3)
    
    # Fine-tuned prediction
    ft_pred, _ = finetuned_classifier.predict_single(query, threshold=0.5)
    
    zs_result = check_prediction(zs_pred, expected)
    ft_result = check_prediction(ft_pred, expected)
    
    if "EXACT" in zs_result:
        zs_correct += 1
    if "EXACT" in ft_result:
        ft_correct += 1
    
    print(f"\nQuery: {query[:65]}{'...' if len(query) > 65 else ''}")
    print(f"Expected: {expected}")
    print(f"  Zero-Shot:  {zs_result:12} {zs_pred}")
    print(f"  Fine-tuned: {ft_result:12} {ft_pred}")

print(f"\n" + "="*70)
print(f"Test Cases Summary:")
print(f"  Zero-Shot:  {zs_correct}/{len(test_cases)} exact matches ({100*zs_correct/len(test_cases):.0f}%)")
print(f"  Fine-tuned: {ft_correct}/{len(test_cases)} exact matches ({100*ft_correct/len(test_cases):.0f}%)")

---
## 7. Summary and Conclusions

### Approaches Compared

| Aspect | Pre-trained (Zero-Shot) | Fine-tuned |
|--------|------------------------|------------|
| Training Data Used | None | Training set only |
| Test Data Seen | Never | Never |
| Model | BART-large-MNLI | DistilBERT |
| Inference Speed | Slower | Faster |
| Domain Knowledge | General | Domain-specific |

### Key Takeaways

1. **No Data Leakage**: Both approaches are evaluated on the same test set that was never seen during training

2. **Zero-Shot Advantages**:
   - No training required
   - Can work immediately with new entity types
   - Good for bootstrapping

3. **Fine-tuned Advantages**:
   - Better performance on domain-specific data
   - Faster inference
   - Learns patterns specific to this dataset

### Metrics Explanation
- **Exact Match**: % of queries where ALL predicted entities exactly match ground truth
- **F1 Score**: Harmonic mean of precision and recall
- **Precision**: Of predicted entities, how many were correct
- **Recall**: Of actual entities, how many were predicted
- **Jaccard Score**: Intersection over union of predicted and true labels
- **Hamming Loss**: Fraction of incorrectly predicted labels (lower is better)

### Open Issues & Future Improvements
1. **Better Zero-Shot**: Try different NLI models or prompt engineering
2. **Ensemble**: Combine zero-shot and fine-tuned predictions
3. **Few-Shot**: Use a small number of examples without full fine-tuning
4. **Threshold Tuning**: Optimize thresholds per entity type

In [ ]:
# Save comparison results
comparison_df.to_csv('approach_comparison_results.csv', index=False)
print("Results saved to approach_comparison_results.csv")